# NeqSim Engineering Study Deliverables — Class A & B Studies

This notebook demonstrates how NeqSim can generate the full set of engineering deliverables 
required for **Class A** (detailed/FEED, ±10-15% accuracy) and **Class B** (concept select, ±15-30%) engineering studies 
per AACE/FEL methodology.

## Deliverable Categories Covered

| # | Discipline | Deliverables | NeqSim Support |
|---|-----------|-------------|----------------|
| 1 | **Process** | Heat & Mass Balance, Equipment List, Utility Summary, Fluid Properties, Emissions | ✅ Full |
| 2 | **Mechanical** | Equipment Datasheets, Vessel/Pipe Sizing, Weight Estimates, Cost Estimates | ✅ Full |
| 3 | **Electrical** | Electrical Load List, Motor Sizing, Cable Sizing, Transformer Sizing, Hazardous Area | ✅ Full |
| 4 | **Safety** | PSV Sizing (API 520), Flare Emissions, Depressuring, Noise, Vibration, CUI Risk | ✅ Full |
| 5 | **Subsea/Wells** | Well Casing Design, SURF Cost, Subsea Equipment | ✅ Full |

## Gaps Identified & Proposed Updates

| Gap | Current Status | Proposed Update |
|-----|---------------|----------------|
| Thermal utilities (cooling water, steam) | Power-focused only | Extend SystemElectricalDesign to track all utilities |
| Auto-PFD generation | Manual export only | Add Graphviz/DOT export from ProcessSystem topology |
| Spare parts inventory | Not tracked | Add InventoryPlanner class |
| Detailed fire case scenarios | Basic thermal relief | Extend fire zone modeling (API 521) |
| Rate-based distillation | Equilibrium only | Out of scope (use external tools) |

---

## Setup — Load NeqSim

In [ ]:
import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except ImportError:
    try:
        import neqsim
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
    print("NeqSim loaded via pip package")

In [ ]:
import jpype
import json
import pandas as pd

# Import NeqSim Java classes
if NEQSIM_MODE == "pip":
    # Thermo
    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

    # Process equipment
    ProcessSystem = jneqsim.process.processmodel.ProcessSystem
    Stream = jneqsim.process.equipment.stream.Stream
    Separator = jneqsim.process.equipment.separator.Separator
    ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
    Compressor = jneqsim.process.equipment.compressor.Compressor
    Cooler = jneqsim.process.equipment.heatexchanger.Cooler
    Heater = jneqsim.process.equipment.heatexchanger.Heater
    HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
    ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
    Mixer = jneqsim.process.equipment.mixer.Mixer
    Pump = jneqsim.process.equipment.pump.Pump

    # Mechanical design
    SystemMechanicalDesign = jpype.JClass("neqsim.process.mechanicaldesign.SystemMechanicalDesign")
    MechanicalDesignReport = jpype.JClass("neqsim.process.mechanicaldesign.MechanicalDesignReport")
    EquipmentDatasheetGenerator = jpype.JClass("neqsim.process.mechanicaldesign.EquipmentDatasheetGenerator")
    EquipmentDesignReport = jpype.JClass("neqsim.process.mechanicaldesign.EquipmentDesignReport")
    CompressorDesignFeasibilityReport = jpype.JClass("neqsim.process.mechanicaldesign.compressor.CompressorDesignFeasibilityReport")
    HeatExchangerDesignFeasibilityReport = jpype.JClass("neqsim.process.mechanicaldesign.heatexchanger.HeatExchangerDesignFeasibilityReport")

    # Electrical design
    SystemElectricalDesign = jpype.JClass("neqsim.process.electricaldesign.system.SystemElectricalDesign")

    # Safety
    SafetyValveMechanicalDesign = jpype.JClass("neqsim.process.mechanicaldesign.valve.SafetyValveMechanicalDesign")
    FlareStack = jpype.JClass("neqsim.process.equipment.flare.FlareStack")

    # Design standards
    NoiseAssessment = jpype.JClass("neqsim.process.mechanicaldesign.designstandards.NoiseAssessment")
    VibrationAssessment = jpype.JClass("neqsim.process.mechanicaldesign.designstandards.VibrationAssessment")
    CUIRiskAssessment = jpype.JClass("neqsim.process.mechanicaldesign.designstandards.CUIRiskAssessment")
    FireProtectionDesign = jpype.JClass("neqsim.process.mechanicaldesign.designstandards.FireProtectionDesign")

    # NEW: Engineering study deliverable classes
    ThermalUtilitySummary = jpype.JClass("neqsim.process.mechanicaldesign.ThermalUtilitySummary")
    ProcessFlowDiagramExporter = jpype.JClass("neqsim.process.processmodel.ProcessFlowDiagramExporter")
    SparePartsInventory = jpype.JClass("neqsim.process.mechanicaldesign.SparePartsInventory")
    AlarmTripScheduleGenerator = jpype.JClass("neqsim.process.mechanicaldesign.AlarmTripScheduleGenerator")

print("All NeqSim classes loaded successfully")

---
## 1. Build a Reference Process (Gas Compression & Cooling Train)

We create a realistic gas processing train that includes all major equipment types:
- Inlet separator
- 2-stage compression with intercooling
- Aftercooler
- Export scrubber

In [ ]:
# ===================================================================
# Create the thermodynamic fluid (natural gas with water)
# ===================================================================
fluid = SystemSrkEos(273.15 + 30.0, 30.0)  # 30°C, 30 bara
fluid.addComponent("nitrogen", 0.02)
fluid.addComponent("CO2", 0.03)
fluid.addComponent("methane", 0.78)
fluid.addComponent("ethane", 0.08)
fluid.addComponent("propane", 0.04)
fluid.addComponent("i-butane", 0.015)
fluid.addComponent("n-butane", 0.015)
fluid.addComponent("n-pentane", 0.005)
fluid.addComponent("water", 0.015)
fluid.setMixingRule("classic")
fluid.setMultiPhaseCheck(True)

# ===================================================================
# Build the process flowsheet
# ===================================================================
process = ProcessSystem()

# --- Feed stream ---
feed = Stream("Feed", fluid)
feed.setFlowRate(50000.0, "kg/hr")
feed.setTemperature(30.0, "C")
feed.setPressure(30.0, "bara")
process.add(feed)

# --- Inlet separator ---
inlet_sep = Separator("V-100 Inlet Separator", feed)
process.add(inlet_sep)

# --- 1st stage compressor ---
comp1 = Compressor("K-100 1st Stage Compressor", inlet_sep.getGasOutStream())
comp1.setOutletPressure(60.0)
comp1.setIsentropicEfficiency(0.75)
process.add(comp1)

# --- Intercooler ---
intercooler = Cooler("E-100 Intercooler", comp1.getOutletStream())
intercooler.setOutTemperature(273.15 + 35.0)
process.add(intercooler)

# --- Interstage scrubber ---
scrubber = Separator("V-101 Interstage Scrubber", intercooler.getOutletStream())
process.add(scrubber)

# --- 2nd stage compressor ---
comp2 = Compressor("K-101 2nd Stage Compressor", scrubber.getGasOutStream())
comp2.setOutletPressure(120.0)
comp2.setIsentropicEfficiency(0.75)
process.add(comp2)

# --- Aftercooler ---
aftercooler = Cooler("E-101 Aftercooler", comp2.getOutletStream())
aftercooler.setOutTemperature(273.15 + 35.0)
process.add(aftercooler)

# --- Export scrubber ---
export_scrubber = Separator("V-102 Export Scrubber", aftercooler.getOutletStream())
process.add(export_scrubber)

# --- Run the process ---
process.run()
print("Process simulation completed successfully")
print(f"  Feed: {float(feed.getFlowRate('kg/hr')):.0f} kg/hr at {float(feed.getPressure('bara')):.1f} bara")
print(f"  Export gas: {float(export_scrubber.getGasOutStream().getFlowRate('kg/hr')):.0f} kg/hr at {float(export_scrubber.getGasOutStream().getPressure('bara')):.1f} bara")
print(f"  Comp 1 power: {float(comp1.getPower('kW')):.0f} kW")
print(f"  Comp 2 power: {float(comp2.getPower('kW')):.0f} kW")

---
## 2. PROCESS DELIVERABLES

### 2.1 Heat & Mass Balance

In [ ]:
# ===================================================================
# Deliverable: Heat & Mass Balance Table
# ===================================================================
# Collect stream data from all equipment
print("="*90)
print("HEAT & MASS BALANCE")
print("="*90)

hmb_rows = []
equipment_list = process.getUnitOperations()
stream_set = set()  # track unique streams

for i in range(equipment_list.size()):
    equip = equipment_list.get(i)
    # Collect inlet streams
    for s in equip.getInletStreams():
        sname = str(s.getName())
        if sname not in stream_set:
            stream_set.add(sname)
            fl = s.getFluid()
            hmb_rows.append({
                "Stream": sname,
                "T (°C)": round(float(s.getTemperature()) - 273.15, 1),
                "P (bara)": round(float(s.getPressure("bara")), 1),
                "Flow (kg/hr)": round(float(s.getFlowRate("kg/hr")), 0),
                "Flow (Sm3/hr)": round(float(s.getFlowRate("Sm3/hr")), 0),
                "Phases": int(fl.getNumberOfPhases()),
                "MW (kg/kmol)": round(float(fl.getMolarMass("kg/mol")) * 1000, 2),
            })
    # Collect outlet streams
    for s in equip.getOutletStreams():
        sname = str(s.getName())
        if sname not in stream_set:
            stream_set.add(sname)
            fl = s.getFluid()
            hmb_rows.append({
                "Stream": sname,
                "T (°C)": round(float(s.getTemperature()) - 273.15, 1),
                "P (bara)": round(float(s.getPressure("bara")), 1),
                "Flow (kg/hr)": round(float(s.getFlowRate("kg/hr")), 0),
                "Flow (Sm3/hr)": round(float(s.getFlowRate("Sm3/hr")), 0),
                "Phases": int(fl.getNumberOfPhases()),
                "MW (kg/kmol)": round(float(fl.getMolarMass("kg/mol")) * 1000, 2),
            })

hmb_df = pd.DataFrame(hmb_rows)
print(hmb_df.to_string(index=False))
print()

In [ ]:
# ===================================================================
# Deliverable: Mass Balance Check
# ===================================================================
print("="*90)
print("MASS BALANCE VERIFICATION")
print("="*90)

mass_balance = process.checkMassBalance("kg/hr")
mb_rows = []
for key in mass_balance.keySet():
    result = mass_balance.get(key)
    mb_rows.append({
        "Equipment": str(key),
        "Mass In (kg/hr)": round(float(result.getMassIn()), 1),
        "Mass Out (kg/hr)": round(float(result.getMassOut()), 1),
        "Error (%)": round(float(result.getRelativeError()) * 100, 4),
    })

mb_df = pd.DataFrame(mb_rows)
print(mb_df.to_string(index=False))
error = process.getMassBalanceError("kg/hr")
print(f"\nOverall mass balance error: {float(error):.6f} kg/hr")

### 2.2 Equipment List

In [ ]:
# ===================================================================
# Deliverable: Equipment List (from SystemMechanicalDesign)
# ===================================================================
print("="*90)
print("EQUIPMENT LIST")
print("="*90)

sys_mech = SystemMechanicalDesign(process)
sys_mech.runDesignCalculation()

equip_list = sys_mech.getEquipmentList()
eq_rows = []
for i in range(equip_list.size()):
    e = equip_list.get(i)
    eq_rows.append({
        "Tag": str(e.getName()),
        "Type": str(e.getEquipmentType()),
        "Design P (barg)": round(float(e.getDesignPressureBarg()), 1),
        "Design T (°C)": round(float(e.getDesignTemperatureC()), 0),
        "Power (kW)": round(float(e.getPowerKW()), 0),
        "Duty (kW)": round(float(e.getDutyKW()), 0),
        "Dry Weight (kg)": round(float(e.getDryWeightKg()), 0),
    })

eq_df = pd.DataFrame(eq_rows)
print(eq_df.to_string(index=False))

In [ ]:
# ===================================================================
# Deliverable: Equipment List as CSV (for import to Excel/database)
# ===================================================================
report = MechanicalDesignReport(process)
report.runDesignCalculations()

csv_output = str(report.generateEquipmentListCSV())
print("EQUIPMENT LIST CSV (first 1000 chars):")
print(csv_output[:1000])

### 2.3 Utility Summary

In [ ]:
# ===================================================================
# Deliverable: Utility Summary
# ===================================================================
print("="*90)
print("UTILITY SUMMARY")
print("="*90)

total_power = float(sys_mech.getTotalPowerRequired())
total_cooling = float(sys_mech.getTotalCoolingDuty())
total_heating = float(sys_mech.getTotalHeatingDuty())

util_data = [
    ["Electrical Power (Compressors/Pumps)", f"{total_power:.0f}", "kW"],
    ["Cooling Duty (Coolers/Condensers)", f"{abs(total_cooling):.0f}", "kW"],
    ["Heating Duty (Heaters/Reboilers)", f"{total_heating:.0f}", "kW"],
]

util_df = pd.DataFrame(util_data, columns=["Utility", "Value", "Unit"])
print(util_df.to_string(index=False))

### 2.4 Fluid Property Tables

In [ ]:
# ===================================================================
# Deliverable: Fluid Property Table (per stream)
# ===================================================================
print("="*90)
print("FLUID PROPERTY TABLE — Export Gas Stream")
print("="*90)

export_stream = export_scrubber.getGasOutStream()
fl = export_stream.getFluid()
fl.initProperties()

props = []
props.append(["Temperature", f"{float(fl.getTemperature()) - 273.15:.1f}", "°C"])
props.append(["Pressure", f"{float(fl.getPressure('bara')):.1f}", "bara"])
props.append(["Molar Mass", f"{float(fl.getMolarMass('kg/mol'))*1000:.2f}", "kg/kmol"])
props.append(["Density", f"{float(fl.getDensity('kg/m3')):.2f}", "kg/m3"])
props.append(["Z-factor", f"{float(fl.getZ()):.4f}", "-"])

if fl.hasPhaseType("gas"):
    gas = fl.getPhase("gas")
    props.append(["Gas Density", f"{float(gas.getDensity('kg/m3')):.2f}", "kg/m3"])
    props.append(["Gas Viscosity", f"{float(gas.getViscosity('kg/msec'))*1e6:.2f}", "µPa·s"])
    props.append(["Gas Therm. Cond.", f"{float(gas.getThermalConductivity('W/mK')):.4f}", "W/m·K"])
    props.append(["Gas Cp", f"{float(gas.getCp('J/kgK')):.1f}", "J/kg·K"])
    props.append(["Gas Enthalpy", f"{float(gas.getEnthalpy('J/mol')):.1f}", "J/mol"])
    props.append(["Gas Entropy", f"{float(gas.getEntropy('J/molK')):.2f}", "J/mol·K"])

prop_df = pd.DataFrame(props, columns=["Property", "Value", "Unit"])
print(prop_df.to_string(index=False))

In [ ]:
# ===================================================================
# Deliverable: Stream Composition Table
# ===================================================================
print("="*90)
print("STREAM COMPOSITION TABLE — Export Gas")
print("="*90)

comp_rows = []
for i in range(fl.getPhase(0).getNumberOfComponents()):
    comp = fl.getPhase(0).getComponent(i)
    comp_rows.append({
        "Component": str(comp.getComponentName()),
        "Mole Fraction": round(float(comp.getx()), 6),
        "MW (g/mol)": round(float(comp.getMolarMass()) * 1000, 2),
    })

comp_df = pd.DataFrame(comp_rows)
print(comp_df.to_string(index=False))

---
## 3. MECHANICAL DESIGN DELIVERABLES

### 3.1 Equipment Datasheets (JSON)

In [ ]:
# ===================================================================
# Deliverable: Equipment Datasheets (auto-generated)
# ===================================================================
print("="*90)
print("EQUIPMENT DATASHEETS (JSON)")
print("="*90)

datasheet_gen = EquipmentDatasheetGenerator(process)
datasheet_gen.setProjectName("Gas Compression Study")
datasheet_gen.setDocumentPrefix("DS")
datasheet_gen.setRevision("A")

all_datasheets_json = str(datasheet_gen.generateAllDatasheets())
datasheets = json.loads(all_datasheets_json)

print(f"Generated {len(datasheets)} equipment datasheets:\n")
for ds in datasheets:
    tag = ds.get("equipmentTag", "N/A")
    eq_type = ds.get("equipmentType", "N/A")
    print(f"  {tag} — {eq_type}")

# Show one datasheet in detail
print("\n--- Sample Datasheet (first compressor) ---")
for ds in datasheets:
    if "Compressor" in str(ds.get("equipmentType", "")):
        print(json.dumps(ds, indent=2, default=str)[:2000])
        break

### 3.2 System Mechanical Design — Weight & Cost Report

In [ ]:
# ===================================================================
# Deliverable: Weight Summary Report
# ===================================================================
print("="*90)
print("WEIGHT SUMMARY")
print("="*90)

# Weight by equipment type
wt_by_type = sys_mech.getWeightByEquipmentType()
wt_rows = []
for key in wt_by_type.keySet():
    wt_rows.append({"Equipment Type": str(key), "Weight (kg)": round(float(wt_by_type.get(key)), 0)})

wt_df = pd.DataFrame(wt_rows)
print("Weight by Equipment Type:")
print(wt_df.to_string(index=False))

# Total weight
print(f"\nTotal Process Weight: {float(sys_mech.getTotalWeight()):.0f} kg")
print(f"Total Plot Space: {float(sys_mech.getTotalPlotSpace()):.0f} m²")

In [ ]:
# ===================================================================
# Deliverable: Complete Mechanical Design Report (JSON)
# ===================================================================
print("="*90)
print("MECHANICAL DESIGN REPORT (JSON Summary)")
print("="*90)

mech_report_json = str(report.toJson())
mech_report = json.loads(mech_report_json)

# Print top-level keys to show report structure
print("Report sections:")
for key in mech_report.keys():
    print(f"  - {key}")

# Print first 3000 chars of the report
print("\nReport preview (first 3000 chars):")
print(json.dumps(mech_report, indent=2, default=str)[:3000])

### 3.3 Compressor Feasibility Report

In [ ]:
# ===================================================================
# Deliverable: Compressor Feasibility Report (API 617 + Cost + Suppliers)
# ===================================================================
print("="*90)
print("COMPRESSOR DESIGN FEASIBILITY REPORT")
print("="*90)

comp_report = CompressorDesignFeasibilityReport(comp1)
comp_report.setDriverType("gas-turbine")
comp_report.setCompressorType("centrifugal")
comp_report.setAnnualOperatingHours(8000)
comp_report.generateReport()

print(f"Verdict: {str(comp_report.getVerdict())}")

comp_json = json.loads(str(comp_report.toJson()))
print(f"\nReport sections: {list(comp_json.keys())}")

# Print key sections
if "feasibilityIssues" in comp_json:
    print("\nFeasibility Issues:")
    for issue in comp_json["feasibilityIssues"]:
        print(f"  [{issue.get('severity', 'N/A')}] {issue.get('description', 'N/A')}")

### 3.4 Combined Equipment Design Report (Mechanical + Electrical + Motor)

In [ ]:
# ===================================================================
# Deliverable: Combined Equipment Design Report
# (Mechanical + Electrical + Motor Design in one report)
# ===================================================================
print("="*90)
print("COMBINED EQUIPMENT DESIGN REPORT — K-100 1st Stage Compressor")
print("="*90)

equip_report = EquipmentDesignReport(comp1)
equip_report.setUseVFD(True)
equip_report.setRatedVoltageV(6600)
equip_report.setHazardousZone(1)
equip_report.setAmbientTemperatureC(40.0)
equip_report.setAltitudeM(100.0)
equip_report.generateReport()

print(f"Verdict: {str(equip_report.getVerdict())}")

edr_json = json.loads(str(equip_report.toJson()))
print(f"\nReport covers {len(edr_json.keys())} design sections:")
for key in edr_json.keys():
    print(f"  - {key}")

print("\nReport preview:")
print(json.dumps(edr_json, indent=2, default=str)[:3000])

---
## 4. ELECTRICAL DESIGN DELIVERABLES

### 4.1 Electrical Load List

In [ ]:
# ===================================================================
# Deliverable: Electrical Load List
# ===================================================================
print("="*90)
print("ELECTRICAL LOAD LIST")
print("="*90)

sys_elec = SystemElectricalDesign(process)
sys_elec.calcDesign()

load_list = sys_elec.getLoadList()
load_json = json.loads(str(load_list.toJson()))

# Display the load items as a table
load_rows = []
if "loadItems" in load_json:
    for item in load_json["loadItems"]:
        load_rows.append({
            "Tag": item.get("tag", "N/A"),
            "Equipment Type": item.get("equipmentType", "N/A"),
            "Rated Power (kW)": round(item.get("ratedPowerKW", 0), 1),
            "Apparent (kVA)": round(item.get("apparentPowerKVA", 0), 1),
            "Power Factor": round(item.get("powerFactor", 0), 2),
            "Duty Cycle": item.get("dutyCycle", "N/A"),
        })

load_df = pd.DataFrame(load_rows)
print(load_df.to_string(index=False))

In [ ]:
# ===================================================================
# Deliverable: System Electrical Summary
# ===================================================================
print("="*90)
print("SYSTEM ELECTRICAL SUMMARY")
print("="*90)

elec_summary = [
    ["Total Process Load", f"{float(sys_elec.getTotalProcessLoadKW()):.0f}", "kW"],
    ["Total Plant Load (incl. utility/UPS)", f"{float(sys_elec.getTotalPlantLoadKW()):.0f}", "kW"],
    ["Main Transformer Size", f"{float(sys_elec.getMainTransformerKVA()):.0f}", "kVA"],
    ["Emergency Generator Size", f"{float(sys_elec.getEmergencyGeneratorKVA()):.0f}", "kVA"],
    ["Overall Power Factor", f"{float(sys_elec.getOverallPowerFactor()):.2f}", "-"],
    ["Connected Load", f"{float(load_list.getTotalConnectedLoadKW()):.0f}", "kW"],
    ["Maximum Demand", f"{float(load_list.getMaximumDemandKW()):.0f}", "kW"],
    ["Required Transformer", f"{float(load_list.getRequiredTransformerKVA()):.0f}", "kVA"],
]

elec_df = pd.DataFrame(elec_summary, columns=["Parameter", "Value", "Unit"])
print(elec_df.to_string(index=False))

---
## 5. SAFETY DELIVERABLES

### 5.1 Safety Valve Sizing (API 520)

In [ ]:
# ===================================================================
# Deliverable: Safety Valve (PSV) Sizing per API 520 / ISO 4126
# ===================================================================
print("="*90)
print("SAFETY VALVE SIZING — API 520 / ISO 4126")
print("="*90)

# Example: PSV on the 1st stage compressor discharge
psv_design = SafetyValveMechanicalDesign(comp1)
psv_design.calcDesign()

psv_json = json.loads(str(psv_design.toJson()))
print("PSV Design Report:")
print(json.dumps(psv_json, indent=2, default=str)[:3000])

### 5.2 Flare Emissions Quantification

In [ ]:
# ===================================================================
# Deliverable: Flare Emissions Quantification
# ===================================================================
print("="*90)
print("FLARE EMISSIONS SUMMARY")
print("="*90)

# Create a flare receiving gas from the inlet separator
flare = FlareStack("Flare", inlet_sep.getGasOutStream())
flare.run()

heat_release = float(flare.getHeatReleaseMW())
emissions = flare.getEmissionsKgPerHr()

print(f"Heat Release: {heat_release:.2f} MW")
print(f"\nEmissions (kg/hr):")

emission_rows = []
for species in emissions.keySet():
    val = float(emissions.get(species))
    emission_rows.append({"Species": str(species), "Rate (kg/hr)": round(val, 2)})

em_df = pd.DataFrame(emission_rows)
print(em_df.to_string(index=False))

### 5.3 Noise & Vibration Assessment

In [ ]:
# ===================================================================
# Deliverable: Noise Assessment (ISO 3744)
# ===================================================================
print("="*90)
print("NOISE ASSESSMENT")
print("="*90)

noise = NoiseAssessment(comp1)
noise_level = float(noise.getSoundPowerLevelDB())
noise_at_1m = float(noise.getSoundPressureLevelAt1mDB())

print(f"Equipment: K-100 1st Stage Compressor")
print(f"Sound Power Level (Lw): {noise_level:.1f} dB(A)")
print(f"Sound Pressure Level at 1m (Lp): {noise_at_1m:.1f} dB(A)")
print(f"Limit (NORSOK S-002): 85 dB(A) at 1m")
print(f"Status: {'PASS' if noise_at_1m <= 85 else 'FAIL — acoustic enclosure required'}")

In [ ]:
# ===================================================================
# Deliverable: Vibration Assessment (ISO 10816-3)
# ===================================================================
print("="*90)
print("VIBRATION ASSESSMENT")
print("="*90)

vibration = VibrationAssessment(comp1)
vib_zone = str(vibration.getVibrationZone())
vib_rms = float(vibration.getVibrationVelocityRMS())

print(f"Equipment: K-100 1st Stage Compressor")
print(f"Vibration Velocity (RMS): {vib_rms:.2f} mm/s")
print(f"ISO 10816-3 Zone: {vib_zone}")
print(f"Status: {'ACCEPTABLE' if vib_zone in ['A', 'B'] else 'UNACCEPTABLE — review mounting/alignment'}")

In [ ]:
# ===================================================================
# Deliverable: CUI (Corrosion Under Insulation) Risk
# ===================================================================
print("="*90)
print("CUI RISK ASSESSMENT (EFC 55)")
print("="*90)

cui = CUIRiskAssessment(intercooler)
risk_level = str(cui.getRiskLevel())
risk_score = float(cui.getRiskScore())

print(f"Equipment: E-100 Intercooler")
print(f"Operating Temperature: {float(intercooler.getOutletStream().getTemperature()) - 273.15:.0f} °C")
print(f"CUI Risk Score: {risk_score:.1f}")
print(f"CUI Risk Level: {risk_level}")
print(f"Recommendation: {'Insulation inspection program required' if risk_level in ['HIGH', 'MEDIUM'] else 'Standard inspection intervals'}")

---
## 6. COMPLETE CLASS A/B DELIVERABLE LIST

### Summary: What NeqSim Delivers vs. What Needs External Tools

In [ ]:
# ===================================================================
# Complete Deliverable Checklist for Class A & B Studies
# ===================================================================
deliverables = [
    # Process
    ["Process", "Heat & Mass Balance", "✅ Full", "ProcessSystem + Stream", "A,B"],
    ["Process", "Mass Balance Verification", "✅ Full", "ProcessSystem.checkMassBalance()", "A,B"],
    ["Process", "Equipment List", "✅ Full", "SystemMechanicalDesign.getEquipmentList()", "A,B"],
    ["Process", "Equipment List (CSV)", "✅ Full", "MechanicalDesignReport.generateEquipmentListCSV()", "A,B"],
    ["Process", "Stream Data Tables", "✅ Full", "Stream.getFluid() properties", "A,B"],
    ["Process", "Fluid Properties", "✅ Full", "SystemInterface.initProperties()", "A,B"],
    ["Process", "Utility Summary (Power)", "✅ Full", "SystemMechanicalDesign", "A,B"],
    ["Process", "Utility Summary (Thermal)", "✅ Full", "SystemMechanicalDesign cooling/heating duty", "A,B"],
    ["Process", "PFD Data (topology)", "✅ Full", "ProcessSystem.getConnections() + getAllElements()", "B"],
    ["Process", "PFD Drawing", "⚠️ Partial", "Manual Graphviz/Visio (topology data available)", "A"],
    ["Process", "P&ID Data", "✅ Full", "DEXPI export (processmodel.dexpi)", "A"],

    # Mechanical
    ["Mechanical", "Equipment Datasheets", "✅ Full", "EquipmentDatasheetGenerator.generateAllDatasheets()", "A,B"],
    ["Mechanical", "Vessel Sizing (Sep/Scrubber)", "✅ Full", "SeparatorMechanicalDesign", "A,B"],
    ["Mechanical", "Pipe/Pipeline Sizing", "✅ Full", "PipelineMechanicalDesign", "A,B"],
    ["Mechanical", "Compressor Feasibility", "✅ Full", "CompressorDesignFeasibilityReport", "A,B"],
    ["Mechanical", "Heat Exchanger Feasibility", "✅ Full", "HeatExchangerDesignFeasibilityReport", "A,B"],
    ["Mechanical", "Weight Report", "✅ Full", "SystemMechanicalDesign", "A,B"],
    ["Mechanical", "CAPEX Estimate", "✅ Full", "Feasibility reports + SURFCostEstimator", "A,B"],
    ["Mechanical", "Design Standards", "✅ Full", "StandardRegistry (20+ standards)", "A,B"],
    ["Mechanical", "Material Selection", "✅ Full", "MaterialPipeDesignStandard / MaterialPlateDesignStandard", "A"],
    ["Mechanical", "Piping Line List", "✅ Full", "ProcessInterconnectionDesign", "A"],
    ["Mechanical", "Combined Mech+Elec+Motor Report", "✅ Full", "EquipmentDesignReport", "A"],

    # Electrical
    ["Electrical", "Electrical Load List", "✅ Full", "ElectricalLoadList", "A,B"],
    ["Electrical", "Motor Sizing", "✅ Full", "ElectricalMotor (IEC 60034)", "A,B"],
    ["Electrical", "VFD Selection", "✅ Full", "VariableFrequencyDrive", "A"],
    ["Electrical", "Cable Sizing", "✅ Full", "ElectricalCable (IEC 60502)", "A"],
    ["Electrical", "Transformer Sizing", "✅ Full", "SystemElectricalDesign", "A,B"],
    ["Electrical", "Emergency Generator", "✅ Full", "SystemElectricalDesign", "A,B"],
    ["Electrical", "Switchgear Sizing", "✅ Full", "Switchgear (IEC 61439)", "A"],
    ["Electrical", "Hazardous Area Classification", "✅ Full", "HazardousAreaClassification (IEC 60079)", "A,B"],

    # Safety
    ["Safety", "PSV Sizing (API 520 / ISO 4126)", "✅ Full", "SafetyValveMechanicalDesign", "A,B"],
    ["Safety", "Flare Emissions", "✅ Full", "FlareStack.getEmissionsKgPerHr()", "A,B"],
    ["Safety", "Flare Heat Release", "✅ Full", "FlareStack.getHeatReleaseMW()", "A,B"],
    ["Safety", "Noise Assessment", "✅ Full", "NoiseAssessment (ISO 3744)", "A,B"],
    ["Safety", "Vibration Assessment", "✅ Full", "VibrationAssessment (ISO 10816-3)", "A,B"],
    ["Safety", "CUI Risk Assessment", "✅ Full", "CUIRiskAssessment (EFC 55)", "A,B"],
    ["Safety", "H2S/Sour Classification", "✅ Full", "DeWaardMilliamsCorrosion", "A,B"],
    ["Safety", "CO2 Corrosion Rate", "✅ Full", "DeWaardMilliamsCorrosion", "A,B"],
    ["Safety", "Depressurization", "✅ Full", "ProcessSystem.runTransient() + PSDValve", "A"],
    ["Safety", "Fire Case Thermal Relief", "⚠️ Partial", "FireProtectionDesign", "A"],
    ["Safety", "Piping Stress Analysis", "✅ Full", "PipingStressAnalysis", "A"],

    # Subsea/Wells
    ["Subsea", "Well Casing Design", "✅ Full", "WellMechanicalDesign (API 5C3)", "A,B"],
    ["Subsea", "SURF Cost Estimate", "✅ Full", "SURFCostEstimator", "A,B"],
    ["Subsea", "Well Cost Estimate", "✅ Full", "WellCostEstimator", "A,B"],
]

deliv_df = pd.DataFrame(deliverables, columns=["Discipline", "Deliverable", "Status", "NeqSim Class/Method", "Study Class"])
print("="*120)
print("COMPLETE DELIVERABLE CHECKLIST — CLASS A & B ENGINEERING STUDIES")
print("="*120)
print(deliv_df.to_string(index=False))

In [ ]:
# Summary statistics
total = len(deliverables)
full = sum(1 for d in deliverables if d[2] == "✅ Full")
partial = sum(1 for d in deliverables if d[2] == "⚠️ Partial")

print(f"\nDELIVERABLE COVERAGE SUMMARY:")
print(f"  Total deliverables identified: {total}")
print(f"  Fully supported:  {full} ({100*full/total:.0f}%)")
print(f"  Partially supported: {partial} ({100*partial/total:.0f}%)")
print(f"  Coverage rate: {100*(full + 0.5*partial)/total:.0f}%")

---
## 7a. Thermal Utility Summary

Aggregates cooling water, steam (LP/MP/HP), and instrument air requirements from the process.

In [ ]:
# Thermal Utility Summary
thermal_util = ThermalUtilitySummary(process)
thermal_util.setCwSupplyTempC(20.0)
thermal_util.setCwReturnTempC(35.0)
thermal_util.calcUtilities()

print("=" * 80)
print("THERMAL UTILITY SUMMARY")
print("=" * 80)
print(f"  Total Cooling Duty:      {float(thermal_util.getTotalCoolingDutyKW()):.1f} kW")
print(f"  Total Heating Duty:      {float(thermal_util.getTotalHeatingDutyKW()):.1f} kW")
print(f"  Cooling Water Flow:      {float(thermal_util.getCoolingWaterFlowM3hr()):.1f} m3/hr")
print(f"  LP Steam:                {float(thermal_util.getLpSteamFlowKghr()):.1f} kg/hr")
print(f"  MP Steam:                {float(thermal_util.getMpSteamFlowKghr()):.1f} kg/hr")
print(f"  HP Steam:                {float(thermal_util.getHpSteamFlowKghr()):.1f} kg/hr")
print(f"  Instrument Air:          {float(thermal_util.getInstrumentAirNm3hr()):.1f} Nm3/hr")
print()

# Individual consumers
consumers = thermal_util.getConsumers()
if consumers.size() > 0:
    data = []
    for i in range(consumers.size()):
        c = consumers.get(i)
        data.append([str(c.getEquipmentName()), str(c.getUtilityType()),
                      float(c.getDutyKW()), float(c.getFlow()), str(c.getFlowUnit())])
    df = pd.DataFrame(data, columns=["Equipment", "Utility", "Duty (kW)", "Flow", "Unit"])
    print("Individual Consumers:")
    print(df.to_string(index=False))

print("\nJSON Report:")
print(str(thermal_util.toJson())[:500], "...")

---
## 7b. Process Flow Diagram Export (Graphviz DOT)

Exports the process topology as a Graphviz DOT string for rendering with `dot` or `neato`.

In [ ]:
# Process Flow Diagram Export
exporter = ProcessFlowDiagramExporter(process)
exporter.setTitle("Gas Compression & Cooling Train PFD")

dot_string = str(exporter.toDot())

print("=" * 80)
print("PROCESS FLOW DIAGRAM — Graphviz DOT Output")
print("=" * 80)
print(dot_string)
print("\nTip: Save to .dot file and render with: dot -Tpng process.dot -o process.png")

---
## 7c. Fire Case Scenarios (API 521 / CCPS)

Extended fire scenarios: pool fire, jet fire, and BLEVE for process equipment.

In [ ]:
# Fire Case Scenario Assessment
print("=" * 80)
print("FIRE CASE SCENARIO ASSESSMENT — API 521 / CCPS")
print("=" * 80)

# Jet fire flame length
jet_length = float(FireProtectionDesign.jetFireFlameLength(5.0, 50000.0))
print(f"\nJet Fire (5 kg/s CH4 release, LHV=50000 kJ/kg):")
print(f"  Flame length:    {jet_length:.1f} m")

# BLEVE assessment for a 50 m3 vessel with 5000 kg LPG
fb_diam = float(FireProtectionDesign.bleveFireballDiameter(5000.0))
fb_dur = float(FireProtectionDesign.bleveFireballDuration(5000.0))
bleve_op = float(FireProtectionDesign.bleveOverpressure(20.0, 50.0, 50.0))
print(f"\nBLEVE (5000 kg LPG, 50 m3 vessel at 20 bara):")
print(f"  Fireball diameter: {fb_diam:.1f} m")
print(f"  Fireball duration: {fb_dur:.1f} s")
print(f"  Overpressure @50m: {bleve_op:.1f} kPa")

# Comprehensive scenario assessment
result = FireProtectionDesign.assessFireScenarios(
    "V-100 HP Separator",  # equipment name
    5000.0,                # inventory kg
    60.0,                  # operating pressure bara
    50.0,                  # vessel volume m3
    10.0,                  # pool diameter m
    2.0,                   # leak rate kg/s
    50000.0,               # LHV kJ/kg
    0.055                  # burning rate kg/(m2*s)
)

print(f"\nComprehensive Fire Scenario for V-100:")
print(f"  Pool Fire HRR:     {float(result.poolFireHeatReleaseKW):.0f} kW")
print(f"  Pool Safe Distance:{float(result.poolFireSafeDistanceM):.1f} m (4.7 kW/m2 threshold)")
print(f"  Jet Flame Length:  {float(result.jetFireFlameLengthM):.1f} m")
print(f"  Jet Safe Distance: {float(result.jetFireSafeDistanceM):.1f} m")
print(f"  BLEVE Fireball:    {float(result.bleveFireballDiameterM):.1f} m dia, {float(result.bleveFireballDurationS):.1f} s")
print(f"  BLEVE Overpressure @50m: {float(result.bleveOverpressureAt50mKPa):.1f} kPa")

print("\nJSON Report:")
print(str(result.toJson()))

---
## 7d. Spare Parts Inventory

Auto-generates recommended spare parts by equipment type with quantity, criticality, and lead time.

In [ ]:
# Spare Parts Inventory
spare_inv = SparePartsInventory(process)
spare_inv.generateInventory()

print("=" * 80)
print("SPARE PARTS INVENTORY")
print("=" * 80)

entries = spare_inv.getEntries()
data = []
for i in range(entries.size()):
    e = entries.get(i)
    data.append([str(e.getEquipmentName()), str(e.getPartName()),
                 int(e.getQuantity()), str(e.getCriticality()), int(e.getLeadTimeWeeks())])

if data:
    df = pd.DataFrame(data, columns=["Equipment", "Spare Part", "Qty", "Criticality", "Lead Time (wks)"])
    print(df.to_string(index=False))

# Critical items
critical = spare_inv.getEntriesByCriticality("Critical")
print(f"\nCritical spare items: {critical.size()}")
print(f"Total items: {entries.size()}")

print("\nJSON Report (first 600 chars):")
print(str(spare_inv.toJson())[:600], "...")

---
## 7e. Noise Propagation with Atmospheric Attenuation (ISO 9613-2)

Far-field noise prediction including frequency-dependent atmospheric absorption.

In [ ]:
# Noise Propagation with ISO 9613-2 Atmospheric Attenuation
print("=" * 80)
print("NOISE PROPAGATION — ISO 9613-2 Atmospheric Attenuation")
print("=" * 80)

# Compressor noise source
swl = float(NoiseAssessment.compressorNoise(3000.0, "centrifugal"))
print(f"\nCompressor SWL: {swl:.1f} dB(A) (3 MW centrifugal)")
print()

# Compare geometric vs atmospheric attenuation
print(f"{'Distance (m)':>12} | {'Geometric Only dB(A)':>22} | {'With Atmos. Atten. dB(A)':>25} | {'Octave Band dB(A)':>18}")
print("-" * 85)
for dist in [10, 25, 50, 100, 200, 500, 1000]:
    spl_basic = float(NoiseAssessment.splAtDistance(swl, float(dist)))
    spl_atten = float(NoiseAssessment.splAtDistanceWithAttenuation(swl, float(dist), 20.0, 70.0))
    spl_octave = float(NoiseAssessment.splAtDistanceOctaveBand(swl, float(dist), 20.0, 70.0))
    print(f"{dist:>12} | {spl_basic:>22.1f} | {spl_atten:>25.1f} | {spl_octave:>18.1f}")

# NORSOK S-002 limit check at 1 m (operator position)
spl_1m = float(NoiseAssessment.splAtDistance(swl, 1.0))
exceeds = bool(NoiseAssessment.exceedsNorsokLimit(spl_1m))
print(f"\nAt 1 m: {spl_1m:.1f} dB(A) — {'EXCEEDS' if exceeds else 'WITHIN'} NORSOK S-002 limit (83 dB(A))")

# Atmospheric absorption at key frequencies
print("\nAtmospheric absorption coefficients (20°C, 70% RH):")
for freq in [63, 125, 250, 500, 1000, 2000, 4000, 8000]:
    alpha = float(NoiseAssessment.atmosphericAbsorption(float(freq), 20.0, 70.0))
    print(f"  {freq:>5} Hz: {alpha:.4f} dB/m  ({alpha*100:.2f} dB/100m)")

---
## 7f. Alarm & Trip Schedule Generator (IEC 61511 / NORSOK I-001)

Auto-generates alarm and trip setpoints from equipment operating conditions and design margins.

In [ ]:
# Alarm & Trip Schedule Generator
alarm_gen = AlarmTripScheduleGenerator(process)
alarm_gen.generate()

print("=" * 80)
print("ALARM & TRIP SCHEDULE")
print("=" * 80)
print(f"Total entries: {int(alarm_gen.getEntryCount())}")
print()

# Display as table
entries = alarm_gen.getEntries()
data = []
for i in range(entries.size()):
    e = entries.get(i)
    data.append([
        str(e.getEquipmentTag()),
        str(e.getInstrumentTag()),
        str(e.getServiceType()),
        str(e.getSetpointType()),
        float(e.getSetpointValue()),
        str(e.getUnit()),
        str(e.getPriority()),
        str(e.getActionType()),
        str(e.getDescription())
    ])

if data:
    df = pd.DataFrame(data, columns=[
        "Equipment", "Instrument", "Service", "Type", "Setpoint", "Unit",
        "Priority", "Action", "Description"
    ])
    print(df.to_string(index=False))

# Entries for separator
sep_entries = alarm_gen.getEntriesForEquipment("HP-Sep")
print(f"\nAlarms for HP-Sep: {sep_entries.size()}")

print("\nJSON Report (first 600 chars):")
print(str(alarm_gen.toJson())[:600], "...")

---
## 7. NEW: Implemented Deliverable Extensions

The following classes have been implemented and are demonstrated below:

| # | Class | Package | Description |
|---|-------|---------|-------------|
| 1 | `ThermalUtilitySummary` | `process.mechanicaldesign` | Aggregates CW, steam, instrument air from process duties |
| 2 | `ProcessFlowDiagramExporter` | `process.processmodel` | Exports ProcessSystem topology to Graphviz DOT |
| 3 | `FireProtectionDesign` (ext.) | `process.mechanicaldesign.designstandards` | Jet fire, BLEVE fireball, overpressure scenarios |
| 4 | `SparePartsInventory` | `process.mechanicaldesign` | Auto-generates recommended spare parts per equipment |
| 5 | `NoiseAssessment` (ext.) | `process.mechanicaldesign.designstandards` | ISO 9613-2 atmospheric attenuation for far-field noise |
| 6 | `AlarmTripScheduleGenerator` | `process.mechanicaldesign` | Auto-generates alarm/trip setpoints from design envelope |

---

## Standards Coverage Summary

| Discipline | Standards Supported |
|-----------|--------------------|
| **Process** | NORSOK P-001/P-002 |
| **Mechanical** | ASME VIII Div.1, ASME B31.3, API 617/610, NORSOK L-001, DNV-ST-F101, ISO 13623 |
| **Electrical** | IEC 60034, IEC 60502, IEC 61439, IEC 60079 (ATEX), IEEE 841, NORSOK E-001 |
| **Safety** | API 520/521, ISO 4126, ISO 3744, ISO 9613-2, ISO 10816-3, NORSOK S-001/S-002, EFC 55, CCPS (BLEVE) |
| **Materials** | NACE MR0175/ISO 15156, API 5L, API 5CT, ISO 3183 |
| **Subsea** | DNV-OS-F101, NORSOK D-010, API 5C3 |
| **Instrumentation** | IEC 61511, NORSOK I-001/I-002 |

---
## 8. Automated Deliverable Generation with StudyClass and EngineeringDeliverablesPackage

The `EngineeringDeliverablesPackage` class automatically generates all required engineering deliverables
based on the study class (A/B/C). It integrates with `FieldDevelopmentDesignOrchestrator` for end-to-end
field development design workflows.

| Study Class | Deliverables Generated |
|-------------|----------------------|
| **CLASS_A** (FEED/Detail) | PFD + Thermal Utilities + Alarm/Trip + Spare Parts + Fire Scenarios + Noise (6 total) |
| **CLASS_B** (Concept/Pre-FEED) | PFD + Thermal Utilities + Fire Scenarios (3 total) |
| **CLASS_C** (Screening) | PFD only (1 total) |

In [ ]:
# 8a. Standalone EngineeringDeliverablesPackage — Class A (full FEED deliverables)
StudyClass = jpype.JClass("neqsim.process.mechanicaldesign.StudyClass")
EngineeringDeliverablesPackage = jpype.JClass("neqsim.process.mechanicaldesign.EngineeringDeliverablesPackage")

# Generate all Class A deliverables from the process system
pkg_a = EngineeringDeliverablesPackage(process, StudyClass.CLASS_A)
pkg_a.generate()

print("=" * 60)
print("  Class A Engineering Deliverables Package")
print("=" * 60)
print(f"Study Class:     {pkg_a.getStudyClass()}")
print(f"Generated:       {pkg_a.isGenerated()}")
print(f"Complete:        {pkg_a.isComplete()}")
print(f"Success Count:   {pkg_a.getSuccessCount()}")
print(f"Failed:          {list(pkg_a.getFailedDeliverables())}")
print()

# Show status of each deliverable
print("Deliverable Status:")
print("-" * 50)
for entry in pkg_a.getStatusMap().entrySet():
    dtype = entry.getKey()
    status = entry.getValue()
    icon = "OK" if status.isSuccess() else "FAIL"
    print(f"  [{icon}] {dtype.getDisplayName():30s} {status.getDurationMs():5d} ms")

# Show PFD DOT excerpt
pfd_dot = str(pkg_a.getPfdDot())
print(f"\nPFD DOT output ({len(pfd_dot)} chars):")
print(pfd_dot[:300], "...")


In [ ]:
# 8b. Class B (Concept study) — reduced deliverable set
pkg_b = EngineeringDeliverablesPackage(process, StudyClass.CLASS_B)
pkg_b.generate()

print("=" * 60)
print("  Class B Engineering Deliverables Package")
print("=" * 60)
print(f"Study Class:     {pkg_b.getStudyClass()}")
print(f"Success Count:   {pkg_b.getSuccessCount()} (out of 3 required)")
print()
print("Deliverable Status:")
print("-" * 50)
for entry in pkg_b.getStatusMap().entrySet():
    dtype = entry.getKey()
    status = entry.getValue()
    icon = "OK" if status.isSuccess() else "FAIL"
    print(f"  [{icon}] {dtype.getDisplayName():30s}")

# Class B should NOT have alarm/trip, spare parts, or noise
print(f"\nAlarm/trip schedule: {pkg_b.getAlarmTripSchedule()}")    # Should be None
print(f"Spare parts:        {pkg_b.getSparePartsInventory()}")     # Should be None
print(f"Noise assessment:   {pkg_b.getNoiseAssessmentJson()}")     # Should be None


### 8c. FieldDevelopmentDesignOrchestrator with Deliverables

The `FieldDevelopmentDesignOrchestrator` now integrates deliverable generation as a workflow step.
When a `StudyClass` is set, the orchestrator automatically generates all required deliverables
after process simulation and mechanical design calculations.

In [ ]:
# 8c. FieldDevelopmentDesignOrchestrator with StudyClass integration
FieldDevelopmentDesignOrchestrator = jpype.JClass(
    "neqsim.process.mechanicaldesign.FieldDevelopmentDesignOrchestrator")
DesignPhase = jpype.JClass("neqsim.process.mechanicaldesign.DesignPhase")

# Create orchestrator for our process
orchestrator = FieldDevelopmentDesignOrchestrator(process, "DEMO-FIELD-001")
orchestrator.setDesignPhase(DesignPhase.FEED)
orchestrator.setStudyClass(StudyClass.CLASS_A)   # Full FEED deliverables

# Run the complete workflow: simulation -> TORG -> mechanical design -> deliverables -> validation
success = orchestrator.runCompleteDesignWorkflow()

print("=" * 70)
print("  FIELD DEVELOPMENT DESIGN ORCHESTRATOR — WORKFLOW RESULT")
print("=" * 70)
print(f"Project ID:      {orchestrator.getProjectId()}")
print(f"Design Phase:    {orchestrator.getDesignPhase()}")
print(f"Study Class:     {orchestrator.getStudyClass()}")
print(f"Run ID:          {orchestrator.getRunId()}")
print()

# Workflow history
print("Workflow Execution Steps:")
print("-" * 50)
for step in orchestrator.getWorkflowHistory():
    icon = "OK" if step.isSuccess() else "FAIL"
    print(f"  [{icon}] {str(step.getStepName()):35s} {step.getDurationMs():5d} ms")

# Validation summary
val = orchestrator.getValidationResult()
print(f"\nValidation: {'PASSED' if val.isValid() else 'FAILED'}")

# Engineering deliverables
deliv = orchestrator.getEngineeringDeliverables()
if deliv is not None:
    print(f"\nEngineering Deliverables: {deliv.getSuccessCount()}/{deliv.getStatusMap().size()} OK")
    print(f"Complete: {deliv.isComplete()}")
    for entry in deliv.getStatusMap().entrySet():
        icon = "OK" if entry.getValue().isSuccess() else "FAIL"
        print(f"  [{icon}] {entry.getKey().getDisplayName()}")


In [ ]:
# 8d. Print the full design report (includes deliverables section)
report = str(orchestrator.generateDesignReport())
print(report)


In [ ]:
# 8e. Full JSON output from the deliverables package
import json

deliv_json = str(deliv.toJson())
parsed = json.loads(deliv_json)

print("=" * 60)
print("  Full Deliverables Package JSON — Summary")
print("=" * 60)
print(f"Study Class:         {parsed['studyClassDisplayName']}")
print(f"Complete:            {parsed['complete']}")
print(f"Success/Total:       {parsed['successCount']}/{parsed['totalRequired']}")
print()

# Show deliverable statuses from the JSON
print("Deliverable Status from JSON:")
for d in parsed['deliverableStatus']:
    icon = "OK" if d['success'] else "FAIL"
    print(f"  [{icon}] {d['deliverable']:30s} ({d['durationMs']} ms)")

# Show availability of each section in the JSON
sections = ['pfdDot', 'thermalUtilities', 'alarmTripSchedule',
            'sparePartsInventory', 'fireScenarios', 'noiseAssessment']
print("\nJSON Sections Present:")
for s in sections:
    present = s in parsed
    size = len(json.dumps(parsed[s])) if present else 0
    print(f"  {'YES' if present else 'NO ':3s}  {s:25s} ({size:,d} chars)")


## 9. Instrument Schedule — Bridging Deliverables and Simulation

The `InstrumentScheduleGenerator` creates an **instrument index** (a standard engineering deliverable) while simultaneously creating **live measurement devices** that are registered on the `ProcessSystem`.

This means one generation step produces:
- **Engineering document**: ISA-5.1 tagged instrument schedule with ranges, alarm setpoints, SIL ratings, and I/O counts
- **Simulation-ready instruments**: Real `MeasurementDeviceInterface` objects with `AlarmConfig`, connected to process streams, queryable by tag

The instruments can then be used for dynamic simulation, alarm testing, plant data integration, or digital twin loops.

| Feature | AlarmTripScheduleGenerator (old) | InstrumentScheduleGenerator (new) |
|---------|----------------------------------|----------------------------------|
| Creates document data | ✓ | ✓ |
| Creates live MeasurementDevice objects | ✗ | ✓ |
| Configures AlarmConfig on devices | ✗ | ✓ |
| Registers devices on ProcessSystem | ✗ | ✓ (optional) |
| ISA-5.1 tag numbering | ✗ | ✓ |
| SIL rating | ✗ | ✓ |
| I/O type classification | ✗ | ✓ |
| Usable for transient simulation | ✗ | ✓ |

In [ ]:
# 9a. Standalone InstrumentScheduleGenerator — create instruments and register on process
import jpype
InstrumentScheduleGenerator = jpype.JClass("neqsim.process.mechanicaldesign.InstrumentScheduleGenerator")
MeasuredVariable = jpype.JClass("neqsim.process.mechanicaldesign.InstrumentScheduleGenerator$MeasuredVariable")

# Count measurement devices BEFORE instrument generation
before_count = len(process.getMeasurementDevices())
print(f"Measurement devices on process BEFORE: {before_count}")

# Create instrument schedule with live device registration
inst_gen = InstrumentScheduleGenerator(process)
inst_gen.setRegisterOnProcess(True)  # Instruments become LIVE in the simulation
inst_gen.generate()

after_count = len(process.getMeasurementDevices())
print(f"Measurement devices on process AFTER:  {after_count}")
print(f"New instruments created: {inst_gen.getInstrumentCount()}")
print()

# Display the instrument schedule
print(f"{'Tag':<10} {'Equipment':<20} {'Service':<35} {'Type':<12} {'Unit':<8} {'Normal':>10} {'Lo':>10} {'Hi':>10} {'LoLo':>10} {'HiHi':>10} {'SIL':<6}")
print("-" * 141)
for entry in inst_gen.getEntries():
    print(f"{str(entry.getTagNumber()):<10} "
          f"{str(entry.getEquipmentTag()):<20} "
          f"{str(entry.getServiceDescription()):<35} "
          f"{str(entry.getMeasuredVariable()):<12} "
          f"{str(entry.getUnit()):<8} "
          f"{float(entry.getNormalValue()):>10.2f} "
          f"{float(entry.getAlarmLow()):>10.2f} "
          f"{float(entry.getAlarmHigh()):>10.2f} "
          f"{float(entry.getTripLow()):>10.2f} "
          f"{float(entry.getTripHigh()):>10.2f} "
          f"{str(entry.getSilRating().getDisplayName()):<6}")

In [ ]:
# 9b. Verify instruments are LIVE on the ProcessSystem — queryable by tag
print("=== Verifying Live Instruments on ProcessSystem ===\n")

# The instruments are now findable by their ISA-5.1 tag on the process
first_entry = inst_gen.getEntries().get(0)
tag = str(first_entry.getTagNumber())
found = process.getMeasurementDeviceByTag(tag)
print(f"Looking up '{tag}' on ProcessSystem: {'FOUND' if found else 'NOT FOUND'}")
if found:
    print(f"  Live measured value: {float(found.getMeasuredValue()):.4f} {str(found.getUnit())}")
    alarm_config = found.getAlarmConfig()
    if alarm_config:
        print(f"  Alarm config: LO={float(alarm_config.getLowLimit()):.2f}, "
              f"HI={float(alarm_config.getHighLimit()):.2f}, "
              f"LOLO={float(alarm_config.getLowLowLimit()):.2f}, "
              f"HIHI={float(alarm_config.getHighHighLimit()):.2f}")

print()

# Show I/O count summary
print("=== I/O Count Summary ===")
pt_count = inst_gen.getCountByType(MeasuredVariable.PRESSURE)
tt_count = inst_gen.getCountByType(MeasuredVariable.TEMPERATURE)
lt_count = inst_gen.getCountByType(MeasuredVariable.LEVEL)
ft_count = inst_gen.getCountByType(MeasuredVariable.FLOW)
print(f"  Pressure transmitters (PT):    {pt_count}")
print(f"  Temperature transmitters (TT): {tt_count}")
print(f"  Level transmitters (LT):       {lt_count}")
print(f"  Flow transmitters (FT):        {ft_count}")
print(f"  Total analog inputs (AI):      {pt_count + tt_count + lt_count + ft_count}")

# Count safety instruments
sil_count = 0
for entry in inst_gen.getEntries():
    if str(entry.getSilRating().getDisplayName()) != "None":
        sil_count += 1
print(f"  Safety instruments (SIL ≥ 1):  {sil_count}")

In [ ]:
# 9c. Integration with EngineeringDeliverablesPackage — instrument schedule as a deliverable
StudyClass = jpype.JClass("neqsim.process.mechanicaldesign.StudyClass")

# Build a fresh process for clean demonstration
pkg_process = ProcessSystem()
pkg_process.add(feed)
pkg_process.add(separator)
pkg_process.add(gas_valve)
pkg_process.add(compressor)
pkg_process.add(afterCooler)
pkg_process.run()

# Generate Class A deliverables — now includes INSTRUMENT_SCHEDULE
EngineeringDeliverablesPackage = jpype.JClass("neqsim.process.mechanicaldesign.EngineeringDeliverablesPackage")
pkg = EngineeringDeliverablesPackage(pkg_process, StudyClass.CLASS_A)
pkg.generate()

print("=== Class A Engineering Deliverables (with Instrument Schedule) ===\n")
print(f"Study class: {str(pkg.getStudyClass())}")
print(f"Total deliverables: {pkg.getSuccessCount()}/{len(pkg.getStatusMap())}")
print(f"Complete: {pkg.isComplete()}")
print()

# Show all deliverables with status
for dtype, status in pkg.getStatusMap().items():
    marker = "✓" if status.isSuccess() else "✗"
    print(f"  {marker} {str(dtype.getDisplayName()):<30} ({int(status.getDurationMs())} ms)")

# Show instrument schedule details from the package
inst = pkg.getInstrumentSchedule()
if inst:
    print(f"\n=== Instrument Schedule from Package ===")
    print(f"  Instruments generated: {inst.getInstrumentCount()}")
    print(f"  Registered on process: {inst.isRegisterOnProcess()}")
    print(f"  Devices queryable by tag: {len(pkg_process.getMeasurementDevices())}")

In [ ]:
# 9d. Show how live instruments can be used for dynamic simulation
import json

print("=== Using Live Instruments for Dynamic Simulation ===\n")
print("After generating the instrument schedule, the devices are LIVE on the ProcessSystem.")
print("They can be used for:")
print("  1. Transient simulation (runTransient reads all measurement devices)")
print("  2. PID controller attachment (setTransmitter on a controller)")
print("  3. Plant data integration (setFieldData with historian tags)")
print("  4. Alarm evaluation (evaluateAlarm during transient runs)")
print()

# Demonstrate: read a live measurement
if inst:
    pt_entries = inst.getEntriesByType(MeasuredVariable.PRESSURE)
    if len(pt_entries) > 0:
        pt_entry = pt_entries.get(0)
        pt_device = pt_entry.getDevice()
        print(f"Live pressure reading from {str(pt_entry.getTagNumber())}:")
        print(f"  Equipment:  {str(pt_entry.getEquipmentTag())}")
        print(f"  Service:    {str(pt_entry.getServiceDescription())}")
        print(f"  Value:      {float(pt_device.getMeasuredValue()):.2f} {str(pt_device.getUnit())}")
        print(f"  Alarm HI:   {float(pt_entry.getAlarmHigh()):.2f} {str(pt_entry.getUnit())}")
        print(f"  Trip HIHI:  {float(pt_entry.getTripHigh()):.2f} {str(pt_entry.getUnit())}")
        print(f"  SIL:        {str(pt_entry.getSilRating().getDisplayName())}")

# Show JSON excerpt
print("\n=== Instrument Schedule JSON (excerpt) ===")
inst_json = json.loads(str(inst.toJson()))
print(f"Total instruments: {inst_json['totalInstruments']}")
print(f"Summary: {json.dumps(inst_json['summary'], indent=2)}")
print(f"I/O count: {json.dumps(inst_json['ioCount'], indent=2)}")
if inst_json.get('instrumentSchedule'):
    print(f"\nFirst 3 entries:")
    for entry in inst_json['instrumentSchedule'][:3]:
        print(f"  {entry['tagNumber']}: {entry['serviceDescription']} "
              f"({entry['instrumentType']}, {entry['silRating']})")